# Step 7: Model Monitoring Setup

**Snowflake Model Monitor** tracks drift and performance.

## Capabilities

| Feature | Description |
|---------|-------------|
| **Drift Detection** | PSI, KL divergence |
| **Performance** | Accuracy, F1 over time |
| **Segmented** | Metrics by segment |
| **Baseline** | Compare to training data |

## Prerequisites

- Run notebooks 01-06 first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session, get_feature_config

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Get Model Version

In [ ]:
MODEL_NAME = config.model.model_name

registry = Registry(
    session,
    database_name=DB,
    schema_name=SCHEMA,
)

model = registry.get_model(MODEL_NAME)
versions = model.versions()
MODEL_VERSION = versions[-1].version_name

print(f"Model: {MODEL_NAME}")
print(f"Version to monitor: {MODEL_VERSION}")

## Understanding Model Monitoring Data Architecture

Model monitors compare **source data** (incoming inference requests) against **baseline data** (training distribution) to detect drift. This requires schema alignment between the two datasets.

### How Autocapture Works

When autocapture is enabled on an inference service, Snowflake automatically logs all requests and responses to `INFERENCE_TABLE()`. The data is stored in a flexible VARIANT format optimized for diverse model signatures:

```
TIMESTAMP | RECORD_ATTRIBUTES (VARIANT) | RESOURCE_ATTRIBUTES (VARIANT)
```

Where `RECORD_ATTRIBUTES` contains the request and response data:
```json
{
  "snow.model_serving.request.data.AGE": 65,
  "snow.model_serving.request.data.BMI": 28.5,
  "snow.model_serving.response.data.output_feature_0": "HIGH",
  ...
}
```

### Data Transformation for Monitoring

To enable column-level drift detection, we transform the data into flat tables with matching schemas:

| Object | Type | Source | Purpose |
|--------|------|--------|---------|
| `INFERENCE_LOGS_VIEW` | View | `INFERENCE_TABLE()` | Extracts features and predictions into individual columns |
| `BASELINE_FOR_MONITORING` | Table | `BASELINE_PATIENT_DATA` | Training distribution with matching column structure |

Both output the same schema for accurate comparison:
```
TIMESTAMP | AGE | GENDER | BMI | ... | RISK_LEVEL
```

> **Note:** The baseline must be a table (not a view) because the monitor clones it internally for drift calculations.

### What Autocapture Provides

| Data | Description |
|------|-------------|
| Request payload | All input features sent to the model |
| Response payload | Model predictions |
| Timestamps | Request/response times |
| Metadata | Model version, service info |

### Monitor Configuration

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `source` | INFERENCE_LOGS_VIEW | View over autocaptured inference data |
| `baseline_table` | BASELINE_FOR_MONITORING | Training distribution for drift comparison |
| `prediction_class_columns` | RISK_LEVEL | Model output to track |
| `segment_columns` | ADMISSION_TYPE, INSURANCE_TYPE | Drill-down dimensions |

### Benefits of Autocapture

1. **Zero instrumentation** - Predictions are captured automatically
2. **Complete coverage** - Every request/response is logged
3. **Historical analysis** - Debug and analyze past inference patterns
4. **Consistent data** - No risk of missing or malformed entries

### Create INFERENCE_LOGS_VIEW

This view flattens the autocaptured inference data from `INFERENCE_TABLE()` into individual columns that the model monitor can analyze for drift.

In [ ]:
feature_config = get_feature_config(config)
computed_features = feature_config["computed_features"]

VIEW_NAME = f"{DB}.{SCHEMA}.INFERENCE_LOGS_VIEW"

computed_numeric = [f for f in computed_features if f != "BMI_CATEGORY"]
computed_numeric_sql = ",\n    ".join([
    f'RECORD_ATTRIBUTES:"snow.model_serving.request.data.{feat}"::FLOAT AS {feat}'
    for feat in computed_numeric
])
bmi_category_sql = 'RECORD_ATTRIBUTES:"snow.model_serving.request.data.BMI_CATEGORY"::VARCHAR AS BMI_CATEGORY'

create_view_sql = f"""
CREATE OR REPLACE VIEW {VIEW_NAME} AS
SELECT
    TIMESTAMP,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE"::NUMBER AS AGE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.GENDER"::VARCHAR AS GENDER,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.BMI"::FLOAT AS BMI,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.HEART_RATE"::NUMBER AS HEART_RATE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.SYSTOLIC_BP"::NUMBER AS SYSTOLIC_BP,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.DIASTOLIC_BP"::NUMBER AS DIASTOLIC_BP,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.TEMPERATURE"::FLOAT AS TEMPERATURE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.RESPIRATORY_RATE"::NUMBER AS RESPIRATORY_RATE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.OXYGEN_SATURATION"::FLOAT AS OXYGEN_SATURATION,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.GLUCOSE_LEVEL"::FLOAT AS GLUCOSE_LEVEL,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.CREATININE"::FLOAT AS CREATININE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.HEMOGLOBIN"::FLOAT AS HEMOGLOBIN,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.WBC_COUNT"::FLOAT AS WBC_COUNT,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.PRIMARY_DIAGNOSIS"::VARCHAR AS PRIMARY_DIAGNOSIS,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.COMORBIDITY_COUNT"::NUMBER AS COMORBIDITY_COUNT,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.ADMISSION_TYPE"::VARCHAR AS ADMISSION_TYPE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.INSURANCE_TYPE"::VARCHAR AS INSURANCE_TYPE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.PREVIOUS_ADMISSIONS"::NUMBER AS PREVIOUS_ADMISSIONS,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.MEDICATION_COUNT"::NUMBER AS MEDICATION_COUNT,
    {computed_numeric_sql},
    {bmi_category_sql},
    RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::VARCHAR AS PREDICTED_RISK_LEVEL
FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
"""

print("Creating inference logs view...")
print(create_view_sql)
session.sql(create_view_sql).collect()

print(f"\nCreated view: {VIEW_NAME}")
print(f"Includes computed features: {computed_features}")
print("\nSample data from auto-captured inference logs:")
session.sql(f"SELECT * FROM {VIEW_NAME} LIMIT 5").show()

### Create BASELINE_FOR_MONITORING Table

This table contains the baseline training distribution with matching column structure to the inference view. The monitor requires:
1. Exact column alignment between source and baseline
2. The baseline to be a **table** (not a view) - the monitor clones it internally

We compute engineered features inline (same logic as the preprocessing notebook) and align data types with the inference view schema.

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.types import FloatType, DoubleType

BASELINE_TABLE = f"{DB}.{SCHEMA}.BASELINE_FOR_MONITORING"
SOURCE_TABLE = f"{DB}.{SCHEMA}.BASELINE_PATIENT_DATA"

raw_df = session.table(SOURCE_TABLE).to_pandas()
raw_df.columns = [c.upper() for c in raw_df.columns]

raw_df["SHOCK_INDEX"] = np.where(
    raw_df["SYSTOLIC_BP"] > 0, raw_df["HEART_RATE"] / raw_df["SYSTOLIC_BP"], np.nan
)
raw_df["PULSE_PRESSURE"] = raw_df["SYSTOLIC_BP"] - raw_df["DIASTOLIC_BP"]
raw_df["BMI_CATEGORY"] = pd.cut(
    raw_df["BMI"],
    bins=[0, 18.5, 25, 30, 35, 100],
    labels=["UNDERWEIGHT", "NORMAL", "OVERWEIGHT", "OBESE", "SEVERELY_OBESE"],
).astype(str)
raw_df["VITAL_SIGNS_SEVERITY"] = (
    (raw_df["HEART_RATE"] > 100).astype(int)
    + (raw_df["HEART_RATE"] < 60).astype(int)
    + (raw_df["RESPIRATORY_RATE"] > 20).astype(int)
    + (raw_df["OXYGEN_SATURATION"] < 94).astype(int) * 2
    + (raw_df["TEMPERATURE"] > 38.0).astype(int)
    + (raw_df["SYSTOLIC_BP"] < 90).astype(int) * 2
    + (raw_df["SYSTOLIC_BP"] > 180).astype(int)
)
raw_df["PREDICTED_RISK_LEVEL"] = raw_df["RISK_LEVEL"]

print(f"Computed engineered features: {computed_features}")
print(f"Total rows: {len(raw_df)}")

### Align Baseline Schema with Inference View

In [ ]:
view_schema = {
    field.name: field.datatype
    for field in session.sql(f"SELECT * FROM {VIEW_NAME} LIMIT 0").schema.fields
}

baseline_df = raw_df[[c for c in view_schema if c in raw_df.columns]].copy()

for col, sf_type in view_schema.items():
    if col in baseline_df.columns and isinstance(sf_type, (FloatType, DoubleType)):
        baseline_df[col] = baseline_df[col].astype("float64")

session.create_dataframe(baseline_df).write.mode("overwrite").save_as_table(BASELINE_TABLE)

print(f"Created baseline table: {BASELINE_TABLE}")
print(f"Columns: {list(baseline_df.columns)}")
print(f"Row count: {len(baseline_df)}")

## Create Model Monitor

Now we create the monitor using SQL that will compare `INFERENCE_LOGS_VIEW` (source) against `BASELINE_FOR_MONITORING` (baseline) to detect:
- **Feature drift**: Changes in input distributions (AGE, BMI, etc.)
- **Prediction drift**: Changes in model output distribution (RISK_LEVEL)

In [ ]:
MONITOR_NAME = f"{MODEL_NAME}_MONITOR"

create_monitor_sql = f"""
CREATE OR REPLACE MODEL MONITOR {MONITOR_NAME} WITH
    MODEL = {MODEL_NAME}
    VERSION = '{MODEL_VERSION}'
    FUNCTION = 'predict'
    SOURCE = {VIEW_NAME}
    WAREHOUSE = {COMPUTE_WAREHOUSE}
    REFRESH_INTERVAL = '1 minute'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = TIMESTAMP
    PREDICTION_CLASS_COLUMNS = ('PREDICTED_RISK_LEVEL')
    BASELINE = {BASELINE_TABLE}
    SEGMENT_COLUMNS = ('ADMISSION_TYPE', 'INSURANCE_TYPE')
"""

print("Creating model monitor...")
print(create_monitor_sql)
session.sql(create_monitor_sql).collect()
print(f"\nModel monitor created: {MONITOR_NAME}")

## Verify Monitor Status

In [ ]:
result = session.sql(f"DESC MODEL MONITOR {MONITOR_NAME}").collect()

if result:
    row = result[0]
    print(f"Monitor: {MONITOR_NAME}")
    print(f"  State: {row['monitor_state']}")
    print(f"  Aggregation Status: {row['aggregation_status']}")
else:
    print("Monitor not found")

## Create Drift Alerts

Create SQL alerts that trigger when drift (PSI) exceeds a threshold on key features. Each alert periodically queries `MODEL_MONITOR_DRIFT_METRIC()` and fires when any drift value exceeds the configured threshold.

In [ ]:
DRIFT_ALERT_COLUMNS = ["AGE", "HEART_RATE", "GLUCOSE_LEVEL"]
DRIFT_THRESHOLD = 0.2
DRIFT_METRIC = "PSI"
ALERT_SCHEDULE = "60 MINUTE"

for col in DRIFT_ALERT_COLUMNS:
    alert_name = f"{MONITOR_NAME}_{col}_DRIFT_ALERT"

    create_alert_sql = f"""
    CREATE OR REPLACE ALERT {alert_name}
        WAREHOUSE = {COMPUTE_WAREHOUSE}
        SCHEDULE = '{ALERT_SCHEDULE}'
        IF (EXISTS (
            SELECT 1
            FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
                '{MONITOR_NAME}',
                '{DRIFT_METRIC}',
                '{col}',
                'DAY',
                DATEADD('day', -1, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
                CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
            ))
            WHERE VALUE > {DRIFT_THRESHOLD}
        ))
        THEN
            CALL SYSTEM$LOG_TRACE('DRIFT_ALERT: {alert_name} triggered. Drift on column {col} exceeded threshold {DRIFT_THRESHOLD} for monitor {MONITOR_NAME}.')
    """

    print(f"Creating drift alert: {alert_name}")
    session.sql(create_alert_sql).collect()
    print(f"  Created: {alert_name} (threshold={DRIFT_THRESHOLD}, schedule={ALERT_SCHEDULE})")

print(f"\nCreated {len(DRIFT_ALERT_COLUMNS)} drift alerts")

## Verify Alerts

In [ ]:
for col in DRIFT_ALERT_COLUMNS:
    alert_name = f"{MONITOR_NAME}_{col}_DRIFT_ALERT"
    result = session.sql(f"DESCRIBE ALERT {alert_name}").collect()
    if result:
        row = result[0]
        print(f"Alert: {row['name']}")
        print(f"  State: {row['state']}")
        print(f"  Schedule: {row['schedule']}")
        print()

## Summary

| Object | Type | Purpose |
|--------|------|---------|
| `INFERENCE_LOGS_VIEW` | View | Flattened autocapture data for monitoring |
| `BASELINE_FOR_MONITORING` | Table | Training distribution for drift comparison |
| `{MODEL_NAME}_MONITOR` | Model Monitor | Tracks drift and prediction changes |
| `*_DRIFT_ALERT` | Alerts | Fire when PSI exceeds threshold on key features |

## Next Step

Continue to **08_streaming_inference.ipynb**